## Query retrieval pipeline 
- augmented generation 
- context + prompt = augmentation 

In [ ]:
# Use refactored src/ pipeline instead of manual Groq wiring
from src.config import get_backend_from_env
from src.pipeline import rag_simple, build_groq_llm
from src.ingestion import load_text_and_pdfs, split_documents
from src.embeddings import EmbeddingManager
from src.retrieval import ChromaRAGRetriever, TypesenseRAGRetriever
from src.vectorstores import ChromaVectorStore, TypesenseVectorStore

: 

In [ ]:
backend = get_backend_from_env("chroma")
print(f"Using backend: {backend}")

# Build retrieval
raw_docs = load_text_and_pdfs()
chunks = split_documents(raw_docs)

if backend == "typesense":
    ts_store = TypesenseVectorStore()
    ts_store.build_from_documents(chunks)
    retriever = TypesenseRAGRetriever(ts_store)
else:
    emb = EmbeddingManager()
    emb.load_model()
    texts = [d.page_content for d in chunks]
    embs = emb.generate_embeddings(texts)

    chroma_store = ChromaVectorStore()
    chroma_store.add_documents(chunks, embs)
    retriever = ChromaRAGRetriever(chroma_store, emb)

llm = build_groq_llm()

answer = rag_simple("What skills is the interviewer looking for?", retriever, llm)
answer